In [ ]:
%%capture
!pip install sentencepiece
!pip install transformers

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration, T5Config, AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.optimization import AdamW
from tqdm import tqdm

In [ ]:
# Define the dataset class
class KeyTextDataset(Dataset):
    def __init__(self, keys, texts, tokenizer):
        self.keys = keys
        self.texts = texts
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        key = self.keys[idx]
        text = self.texts[idx]
        key_encoding = self.tokenizer(
            key,
            max_length=512,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            add_special_tokens=True,
            return_tensors='pt'
        )

        text_encoding = self.tokenizer(
            text,
            max_length=512,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            add_special_tokens=True,
            return_tensors='pt'
        )
        input_ids = key_encoding['input_ids'].squeeze()
        attention_mask = key_encoding['attention_mask'].squeeze()

        # print(text_encoding)

        labels = text_encoding['input_ids'].squeeze()
        labels[labels == 0] = -100
        labels_attention_mask = text_encoding['attention_mask'].squeeze()

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'labels_attention_mask':labels_attention_mask,
            'text': text
        }

# Function to train the model
def train_model(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0

    # progress_bar = tqdm(enumerate(dataloader), total=len(dataloader))
    for step, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        labels_attention_mask = batch['labels_attention_mask'].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_attention_mask=labels_attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

        # progress_bar.set_description(f"Train Loss: {loss.item():.4f}")

    return total_loss / len(dataloader)

# Function to validate the model
def validate_model(model, dataloader, device):
    model.eval()
    total_loss = 0

    # progress_bar = tqdm(enumerate(dataloader), total=len(dataloader))
    for step, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        labels_attention_mask = batch['labels_attention_mask'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_attention_mask=labels_attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        # progress_bar.set_description(f"Train Loss: {loss.item():.4f}")

    return total_loss / len(dataloader)

# Function to save the trained model and tokenizer
def save_model(model, tokenizer, output_dir):
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"Model and tokenizer saved to '{output_dir}'")

# Function to load the saved model and tokenizer
def load_model(output_dir):
    model = AutoModelForSeq2SeqLM.from_pretrained(output_dir)
    tokenizer = AutoTokenizer.from_pretrained(output_dir)
    print(f"Model and tokenizer loaded from '{output_dir}'")
    return model, tokenizer

# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

loading_model_dir = "./Model/bnT5ModelV26"
# loading_model_dir = "./Model/ModelV26"
loaded_model, loaded_tokenizer = load_model(loading_model_dir)
loaded_model.to(device)

In [ ]:
# Function to generate text given a key
def generate_text(key):
    input_ids = loaded_tokenizer.encode(key, return_tensors='pt',add_special_tokens=True).to(device)

    with torch.no_grad():
      outputs = loaded_model.generate(
          input_ids=input_ids,
          max_length =64,
          # max_new_tokens = 64,
          # num_beams =2,
          # num_beams = 1, # For Greedy
          # early_stopping =True,
          num_return_sequences = 1,
          temperature = 0.3,
          # top_k= 50,
          top_p= 0.95,
          do_sample=True,
          # do_sample=False, # For Greedy
          repetition_penalty= 2.5,
          length_penalty= 1.0)

    # print(outputs)

    preds = [loaded_tokenizer.decode(g,skip_special_tokens=True,clean_up_tokenization_spaces=True) for g in outputs]
    generated_text = preds[0]

    return generated_text

def predict(key):
  return generate_text(key)

In [ ]:
key = "ও রক্ষণাবেক্ষণের শেষের বর্ষা বা দিকে হয় অন্যদিকে"
predict(key) # mT5: অন্যদিকে বর্ষা ও বন্যার রক্ষণাবেক্ষণের কাজ শেষের দিকে শুরু হয় না।

'অন্যদিকে বর্ষা ও শীতের শেষের দিকে রক্ষণাবেক্ষণের কাজ শুরু হয়।'

In [ ]:
# Function to generate text given a key
def generate_text_mod(key, forceWord):
    input_ids = loaded_tokenizer.encode(key, return_tensors='pt',add_special_tokens=True).to(device)
    key2 = [forceWord]
    force_words_ids = loaded_tokenizer(key2, add_special_tokens=False).input_ids
    # force_words_ids = [loaded_tokenizer(key, add_special_tokens=False).input_ids]

    with torch.no_grad():
      outputs = loaded_model.generate(
      input_ids=input_ids,
    #   bos_token_id = loaded_tokenizer.pad_token_id,
      force_words_ids=force_words_ids,
      max_length =64,
      num_beams=4,
      num_return_sequences=1,
      no_repeat_ngram_size=1,
      remove_invalid_values=True,
      do_sample=True,
      repetition_penalty= 2.5,
      length_penalty= 1.0)

    preds = [loaded_tokenizer.decode(g,skip_special_tokens=True,clean_up_tokenization_spaces=True) for g in outputs]

    generated_text = preds[0]
    return generated_text


def predict2(key, forceWord):
  return generate_text_mod(key, forceWord)

In [ ]:
missingWord = "বা"
print(f'\n{predict2(key, missingWord)}') # mT5: অন্যদিকে বর্ষা শুরুর পর থেকে তাঁদের প্রশিক্ষণ ও পরিবেশবান্ধব সড়ক-মহাব্যবস্থাপনা বা রাস্তাঘাট–সংলগ্ন জায়গাগুলোও (রক্ষণাবেক্ষণের) সময়সীমা বাড়ানো হয়।


অন্যদিকে বর্ষা বা শরতের শেষের দিকে রক্ষণাবেক্ষণের কাজ শুরু হয়।
